In [2]:
import nltk
import string
from datasets import load_dataset
import os

c:\Users\ysnxlmted\Projects\documents_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Создание папки для nltk данных, если её нет
nltk_data_dir = os.path.expanduser('../nltk_data')
if not os.path.exists(nltk_data_dir):
    os.makedirs(nltk_data_dir)

# Добавление пути в nltk
nltk.data.path.append(nltk_data_dir)

# Загрузка всех необходимых ресурсов
resources = ['punkt', 'wordnet', 'omw-1.4', 'punkt_tab', 
             'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'stopwords']

for resource in resources:
    try:
        nltk.download(resource, download_dir=nltk_data_dir, quiet=False)
    except:
        print(f"Ресурс {resource} уже загружен или произошла ошибка")


[nltk_data] Downloading package punkt to ../nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to ../nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to ../nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to ../nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to ../nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Функция для лемматизации с удалением стоп слов

In [ ]:
from nltk.corpus import wordnet
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

# Применение лемматизации
lemmatizer = nltk.WordNetLemmatizer()
# Функция предобработки с лемматизацией
def preprocess_with_lemmatization(text):
    # Приведение к нижнему регистру
    text = text.lower()
    # Токенизация
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)

    lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token, tag in tagged
                        if token not in string.punctuation]
    

    return remove_stopwords(lemmatized_tokens)

# Функция для представления в виде векторов 

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

def vectorizer1(lemmatized_result):
    '''
    сразу засовываем лемматизированные тексты
    '''

    docs_as_strings = [''.join(tokens) for tokens in lemmatized_result]
    vectorizer = CountVectorizer(binary=True)
    
    X = vectorizer.fit_transform(docs_as_strings)
    vocab = vectorizer.get_feature_names_out()
    
    return X, vocab

def vectorizer2(raw_docs):
    '''
    подаем просто тексты, 
    внутри векторизатора используем наш токенезетор
    '''

    vectorizer = CountVectorizer(
        binary=True,
        tokenizer=preprocess_with_lemmatization,
        lowercase=False,
        token_pattern=None
    )
    
    X = vectorizer.fit_transform(raw_docs)
    vocab = vectorizer.get_feature_names_out()

    return X, vocab

def vectorizer3(raw_docs):
    '''
    полагаемся на исходный векторизатор
    '''

    vectorizer = CountVectorizer(binary=True)
    
    X = vectorizer.fit_transform(raw_docs)
    vocab = vectorizer.get_feature_names_out()

    return X, vocab

In [6]:
def vectorizers_outp(lemmatized_result, other_texts):
    vec1 = vectorizer1(lemmatized_result)
    vec2 = vectorizer2(other_texts)
    vec3 = vectorizer3(other_texts)
    print("СРАВНЕНИЕ ВЕКТОРИЗАТОРОВ")

    print("1st")
    print(f"Словарь: {vec1[1]}, \n {vec1[0].toarray()}")

    print("2st")
    print(f"Словарь: {vec2[1]}, \n {vec2[0].toarray()}")

    print("3st")
    print(f"Словарь: {vec3[1]}, \n {vec3[0].toarray()}")

In [7]:
# Загрузка датасета Emotion (HuggingFace)
dataset = load_dataset("emotion")

In [8]:
# Посмотрим другие примеры
other_examples = dataset["train"][:5]
other_texts = other_examples["text"]

for i, text in enumerate(other_texts):
    print(f"Пример {i+1}:")
    print(text)
    print()

Пример 1:
i didnt feel humiliated

Пример 2:
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake

Пример 3:
im grabbing a minute to post i feel greedy wrong

Пример 4:
i am ever feeling nostalgic about the fireplace i will know that it is still on the property

Пример 5:
i am feeling grouchy



In [9]:
lemmatized_result = [preprocess_with_lemmatization(text) for text in other_texts]

for i in range(len(other_texts)):
    print(f"Пример {i+1}:")
    print(f"Lemmatizing: {lemmatized_result[i]}, length: {len(lemmatized_result[i])}")
    print()

Пример 1:
Lemmatizing: ['didnt', 'feel', 'humiliate'], length: 3

Пример 2:
Lemmatizing: ['go', 'feel', 'hopeless', 'damned', 'hopeful', 'around', 'someone', 'care', 'awake'], length: 9

Пример 3:
Lemmatizing: ['im', 'grab', 'minute', 'post', 'feel', 'greedy', 'wrong'], length: 7

Пример 4:
Lemmatizing: ['ever', 'feel', 'nostalgic', 'fireplace', 'know', 'still', 'property'], length: 7

Пример 5:
Lemmatizing: ['feel', 'grouchy'], length: 2



In [10]:
vectorizers_outp(lemmatized_result, other_texts)

СРАВНЕНИЕ ВЕКТОРИЗАТОРОВ
1st
Словарь: ['didntfeelhumiliate' 'everfeelnostalgicfireplaceknowstillproperty'
 'feelgrouchy' 'gofeelhopelessdamnedhopefularoundsomeonecareawake'
 'imgrabminutepostfeelgreedywrong'], 
 [[1 0 0 0 0]
 [0 0 0 1 0]
 [0 0 0 0 1]
 [0 1 0 0 0]
 [0 0 1 0 0]]
2st
Словарь: ['around' 'awake' 'care' 'damned' 'didnt' 'ever' 'feel' 'fireplace' 'go'
 'grab' 'greedy' 'grouchy' 'hopeful' 'hopeless' 'humiliate' 'im' 'know'
 'minute' 'nostalgic' 'post' 'property' 'someone' 'still' 'wrong'], 
 [[0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [1 1 1 1 0 0 1 0 1 0 0 0 1 1 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0 0 1 1 0 0 0 0 1 0 1 0 1 0 0 0 1]
 [0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 0 1 0 1 0 1 0]
 [0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]]
3st
Словарь: ['about' 'am' 'and' 'around' 'awake' 'being' 'can' 'cares' 'damned'
 'didnt' 'ever' 'feel' 'feeling' 'fireplace' 'from' 'go' 'grabbing'
 'greedy' 'grouchy' 'hopeful' 'hopeless' 'humiliated' 'im' 'is' 'it'
 'just' 'know' 'minu

In [11]:
# Загрузка 20 Newsgroups с 4 классами
dataset = load_dataset("SetFit/20_newsgroups", split="train")

categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x"
]

filtered_texts = [
    example["text"]
    for example in dataset
    if example["label_text"] in categories
][:5]

filtered_texts

Repo card metadata block was not found. Setting CardData to empty.


["A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if you have done the clock upgrade and haven't answered this\npoll. Thanks.",
 'well folks, my mac plus finally gave up the ghost this weekend after\nstarting life as a 512k way back in 1985.  sooo, i\'m in the market for a\nnew machine a bit sooner than i intended to be...\n\ni\'m looking into picking up a powerbook 160 or maybe 180 and have a bunch\nof questions that (hopefully) somebody can answer:\n\n* does anybody know any dirt on when the next round of powerbook\nintroductions are expected?  i\'d heard the 185c was supposed

In [12]:
for i, text in enumerate(filtered_texts):
    print(f"Пример {i+1}:")
    print(text)
    print()

Пример 1:
A fair number of brave souls who upgraded their SI clock oscillator have
shared their experiences for this poll. Please send a brief message detailing
your experiences with the procedure. Top speed attained, CPU rated speed,
add on cards and adapters, heat sinks, hour of usage per day, floppy disk
functionality with 800 and 1.4 m floppies are especially requested.

I will be summarizing in the next two days, so please add to the network
knowledge base if you have done the clock upgrade and haven't answered this
poll. Thanks.

Пример 2:
well folks, my mac plus finally gave up the ghost this weekend after
starting life as a 512k way back in 1985.  sooo, i'm in the market for a
new machine a bit sooner than i intended to be...

i'm looking into picking up a powerbook 160 or maybe 180 and have a bunch
of questions that (hopefully) somebody can answer:

* does anybody know any dirt on when the next round of powerbook
introductions are expected?  i'd heard the 185c was supposed to 

In [13]:
lemmatized_result = [preprocess_with_lemmatization(text) for text in filtered_texts]

for i in range(len(filtered_texts)):
    print(f"Пример {i+1}:")
    print(f"Lemmatizing: {lemmatized_result[i]}, length: {len(lemmatized_result[i])}")
    print()

Пример 1:
Lemmatizing: ['fair', 'number', 'brave', 'soul', 'upgrade', 'si', 'clock', 'oscillator', 'share', 'experience', 'poll', 'please', 'send', 'brief', 'message', 'detail', 'experience', 'procedure', 'top', 'speed', 'attain', 'cpu', 'rat', 'speed', 'add', 'card', 'adapter', 'heat', 'sink', 'hour', 'usage', 'per', 'day', 'floppy', 'disk', 'functionality', '800', '1.4', 'floppy', 'especially', 'request', 'summarize', 'next', 'two', 'day', 'please', 'add', 'network', 'knowledge', 'base', 'clock', 'upgrade', "n't", 'answer', 'poll', 'thanks'], length: 56

Пример 2:
Lemmatizing: ['well', 'folk', 'mac', 'plus', 'finally', 'give', 'ghost', 'weekend', 'start', 'life', '512k', 'way', 'back', '1985.', 'sooo', "'m", 'market', 'new', 'machine', 'bit', 'sooner', 'intend', '...', "'m", 'look', 'pick', 'powerbook', '160', 'maybe', '180', 'bunch', 'question', 'hopefully', 'somebody', 'answer', 'anybody', 'know', 'dirt', 'next', 'round', 'powerbook', 'introduction', 'expect', "'d", 'hear', '185c',

In [14]:
vectorizers_outp(lemmatized_result, filtered_texts)

СРАВНЕНИЕ ВЕКТОРИЗАТОРОВ
1st
Словарь: ['1208' '12mb' '15mb' '1chip' '1controlerchiprangeindeed0'
 '1existpcusesetscsi' '1go' '1interfacedrivemachinescsi' '1interfacethink'
 '1maybescsi' '1mode4' '1reach10mb' '1scsi' '20fastide' '20mb' '216' '232'
 '28' '2chipapplesalespersonsay' '2controllerchip4' '2controllerchip8'
 '2controllerchipscsi' '39' '44'
 '4floppyespeciallyrequestsummarizenexttwodaypleaseaddnetworkknowledgebaseclockupgraden'
 '5mb' '6info' '6mb'
 '96scsifactpostnewsgroupmacibminfosheetavailableftpsumex' 'aim'
 'althoughscsitwicefastesdi'
 'anybodyheardrumorpricedroppowerbooklinelikeoneduo' 'bit'
 'bitmodemuchfastertruescsi' 'bitnoteincreasespeedmacquadrauseversionscsi'
 'compareversion'
 'compressedfileunlessboardinstalsincestacproductseemunlikelyholeautodoubler'
 'correctscsi'
 'dayworthtakedisksizemoneyhitgetactivedisplayrealizerealsubjectivequestion'
 'dhear185csupposemakeappearence'
 'diskdoublerrelateboardfixsadmakereluctantbuystac'
 'displayyealookgreatstore' 'dlikeget